In [0]:
dbutils.widgets.text("login", "pkustra555")
dbutils.widgets.text("catalog", "dbr_dev")
dbutils.widgets.text("dataset", "social_media_addiction_mental_wellbeing")

In [0]:
catalog = dbutils.widgets.get("catalog")
login = dbutils.widgets.get("login")
dataset = dbutils.widgets.get("dataset")

In [0]:
bronze_schema = f"{login}_bronze"
base_path = f"/Volumes/{catalog}/{bronze_schema}/streaming_lab"

input_path = f"{base_path}/input"
schema_path = f"{base_path}/schema"
checkpoint_path = f"{base_path}/checkpoints/autoloader_ingestion"

target_table = f"{catalog}.{bronze_schema}.{dataset}_bronze"

In [0]:

stream_df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("header", True)
        .load(input_path)
)

In [0]:
query = (
    stream_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(target_table)
)

query.awaitTermination()

In [0]:
display(dbutils.fs.ls(checkpoint_path))

In [0]:
print(query.status)

In [0]:
display(spark.table(target_table))